# NIBFS reproducibility workflow — primary k=20

This notebook provides the reference raw-to-core analysis workflow used for the manuscript.

The code remains visible cell by cell, and every output appears below the
corresponding cell. No previous paper result is embedded.

## Analysis settings

- Primary frozen panel: **top-20 NIBFS genes**
- Panel-size sensitivity: **k = 10, 20, 30, and 50**
- Complete top-20 panel is required in GSE15852/GPL96
- Five-fold supervised feature selection is training-restricted
- Comparators: DEG-only, mRMR, and LASSO
- Classifiers: LR, RF, and LightGBM
- Rank-weight sensitivity: 0.25, 0.50, and 0.75
- Enrichment, STRING/PPI, KM Plotter, and KAN bridge are included
- Repeated 5-fold × 10 is optional and disabled
- LOCO is eligibility-only; full LOCO performance is not claimed

Use only values generated inside one complete timestamped run folder.

## 1. Configuration and Google Drive

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import os
import sys
import time
import shutil
import warnings
import json
import hashlib
import zipfile

from google.colab import drive
drive.mount('/content/drive')

markers = list(
    Path('/content/drive/MyDrive').rglob(
        'NIBFS_REPRODUCIBILITY_PACKAGE.marker'
    )
)
if len(markers) != 1:
    raise RuntimeError(
        'Expected exactly one extracted package. '
        f'Found {len(markers)} marker files: {markers}'
    )

PACKAGE_DIR = markers[0].parent.resolve()
os.chdir(PACKAGE_DIR)
sys.path.insert(0, str(PACKAGE_DIR))

RUN_DIR = PACKAGE_DIR / 'runs' / (
    'NIBFS_RAW_RUN_' + datetime.now().strftime('%Y%m%d_%H%M%S')
)
RUN_DIR.mkdir(parents=True, exist_ok=False)

print('PACKAGE_DIR:', PACKAGE_DIR)
print('RUN_DIR:', RUN_DIR)
print('src exists:', (PACKAGE_DIR / 'src').is_dir())
print('config exists:', (PACKAGE_DIR / 'config.yaml').exists())

## 2. Install and verify the Colab environment

In [ ]:
import subprocess
subprocess.check_call([
    sys.executable,
    str(PACKAGE_DIR / 'install_environment_colab.py')
])

## 3. Imports, directories, and fixed parameters

In [ ]:
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml

from sklearn.model_selection import StratifiedKFold, train_test_split

from src.download_utils import (
    HGNC_COMPLETE_SET_URL,
    download_first_available,
    download_if_missing,
    geo_annotation_url,
    geo_series_urls,
    string_reference_urls,
)
from src.hgnc import HGNCResolver
from src.geo_io import (
    parse_series_matrix,
    create_sample_metadata,
    label_discovery_sample,
    read_geo_annotation,
)
from src.preprocessing import (
    conditional_log2_probe_table,
    highest_variance_probe_per_gene,
    quantile_normalize_samples,
    combat_harmonize,
    filter_bottom_variance,
    binary_labels,
)
from src.quality_control import (
    expression_distribution_summary,
    pca_table,
    sample_correlation_audit,
    missing_value_summary,
)
from src.ppi import (
    read_string_protein_to_gene,
    build_gene_edges_and_degree,
    final_panel_subnetwork,
)
from src.feature_selection import (
    prepare_ppi_rank_table,
    select_top_k,
    pairwise_jaccard_summary,
)
from src.workflow import (
    dirs,
    rankings,
    panels,
    load_external,
    _long_panels,
    _full_panels,
    _source_subset,
)
from src.modeling import (
    create_models,
    fit_predict_panels,
    metrics_from_predictions,
    summarize_cv,
    add_auc_confidence_intervals,
    calibration_table,
)
from src.statistics import (
    derive_youden_thresholds,
    stability_statistical_tests,
    cv_auc_statistical_tests,
)
from src.enrichment import run_enrichment, standardize_enrichment
from src.sensitivity import run_weight_sensitivity
from src.kmplotter import load_kmplotter_csv
from src.reporting import (
    choose_compact_heatmap_samples,
    clustered_heatmap_orders,
    build_kan_bridge,
    write_json,
)
from src.figures import (
    plot_pca_before_after,
    plot_expression_distributions,
    plot_missing_values,
    plot_correlation_heatmap,
    plot_outlier_audit,
    plot_preprocessing_qc_overview,
    plot_volcano,
    plot_clustered_heatmap,
    plot_stability_composite,
    plot_gene_occurrence_heatmap,
    plot_cv_performance,
    plot_sensitivity,
    plot_rank_landscape,
    plot_roc,
    plot_heldout_composite,
    plot_external_validation,
    plot_ppi_network,
    plot_enrichment,
    plot_biological_interpretation,
    plot_km_forest,
    plot_study_workflow,
    plot_graphical_abstract,
    plot_top_panel_barplot,
)

cfg = yaml.safe_load((PACKAGE_DIR / 'config.yaml').read_text())
FINAL_K = int(cfg['project']['final_k'])
K_VALUES = list(map(int, cfg['project']['sensitivity_k']))
RANDOM_STATE = int(cfg['project']['random_state'])
CV_FOLDS = int(cfg['project']['cv_folds'])

if FINAL_K not in K_VALUES:
    raise ValueError('FINAL_K must appear in K_VALUES.')

d = dirs(RUN_DIR)
TABLE_DIR = RUN_DIR / 'results' / 'main' / 'tables'
FIGURE_DIR = RUN_DIR / 'results' / 'main' / 'figures'
MODEL_DIR = RUN_DIR / 'results' / 'main' / 'models'
SUPP_TABLE_DIR = RUN_DIR / 'results' / 'main' / 'supplementary_tables'
SUPP_FIGURE_DIR = RUN_DIR / 'results' / 'main' / 'supplementary_figures'
KAN_DIR = RUN_DIR / 'results' / 'downstream_KAN' / 'frozen_inputs'
LOG_DIR = RUN_DIR / 'results' / 'main' / 'logs'

for path in [
    TABLE_DIR, FIGURE_DIR, MODEL_DIR,
    SUPP_TABLE_DIR, SUPP_FIGURE_DIR, KAN_DIR, LOG_DIR
]:
    path.mkdir(parents=True, exist_ok=True)

d['tables'] = TABLE_DIR
d['figures'] = FIGURE_DIR
d['models'] = MODEL_DIR
d['supp'] = SUPP_TABLE_DIR
d['logs'] = LOG_DIR

shutil.copy2(PACKAGE_DIR / 'config.yaml', RUN_DIR / 'config.yaml')
shutil.copy2(
    PACKAGE_DIR / 'data_accession_list.csv',
    RUN_DIR / 'data_accession_list.csv'
)

display(pd.DataFrame([cfg['project']]))
display(pd.DataFrame([cfg['preprocessing']]))
display(pd.DataFrame([cfg['models']]))

# Final paper decision gate.
if FINAL_K != 20:
    raise RuntimeError(
        f"This final package requires FINAL_K=20, found {FINAL_K}."
    )

if K_VALUES != [10, 20, 30, 50]:
    raise RuntimeError(
        "Sensitivity sizes must be [10, 20, 30, 50]."
    )

if not bool(
    cfg["external_validation"]["require_complete_frozen_panel"]
):
    raise RuntimeError(
        "Complete top-20 external coverage must be required."
    )

print("FINAL DECISION CHECK: PASS")
print("Primary panel: k=20")
print("Sensitivity: k=10,20,30,50")
print("Complete external top-20 coverage: REQUIRED")


## 4. Runtime tracker

In [ ]:
runtime_rows = []
pipeline_wall_start = time.perf_counter()

def start_stage(stage_name):
    print('\n' + '=' * 88)
    print('START:', stage_name)
    print('=' * 88)
    return stage_name, time.perf_counter()

def finish_stage(token):
    stage_name, started = token
    seconds = time.perf_counter() - started
    runtime_rows.append({
        'Stage': stage_name,
        'Seconds': seconds,
        'Minutes': seconds / 60,
        'Finished_UTC': datetime.now(timezone.utc).isoformat(),
    })
    pd.DataFrame(runtime_rows).to_csv(
        TABLE_DIR / 'computation_time_summary.csv',
        index=False
    )
    print(
        f'FINISH {stage_name}: '
        f'{seconds:.2f} seconds ({seconds/60:.2f} minutes)'
    )

## 5. Discovery dataset list

In [ ]:
accession_table = pd.read_csv(PACKAGE_DIR / 'data_accession_list.csv')
discovery_table = accession_table[
    accession_table['Role'].str.startswith('Discovery')
].copy()
external_table = accession_table[
    accession_table['Role'].str.startswith('External')
].copy()

discovery_table.to_csv(
    TABLE_DIR / 'dataset_composition_table.csv',
    index=False
)
discovery_table.to_csv(
    SUPP_TABLE_DIR / 'Table_S1_discovery_cohorts.csv',
    index=False
)

display(discovery_table)
display(external_table)

## 6. Download HGNC, STRING, and GPL570 annotation references

In [ ]:
token = start_stage('Reference downloads')

hgnc_path = download_if_missing(
    HGNC_COMPLETE_SET_URL,
    d['ref'] / 'hgnc_complete_set.txt'
)
resolver = HGNCResolver.from_complete_set(hgnc_path)

string_urls = string_reference_urls(
    int(cfg['ppi']['species']),
    str(cfg['ppi']['string_version'])
)
string_info_path = download_if_missing(
    string_urls['protein_info'],
    d['ref'] / Path(string_urls['protein_info']).name
)
string_links_path = download_if_missing(
    string_urls['protein_links'],
    d['ref'] / Path(string_urls['protein_links']).name
)

gpl570_path = download_if_missing(
    geo_annotation_url('GPL570'),
    d['geo'] / 'GPL570.annot.gz'
)
probe_map = read_geo_annotation(gpl570_path, resolver)
probe_map.to_csv(
    TABLE_DIR / 'GPL570_probe_to_HGNC_mapping.csv',
    index=False
)

reference_summary = pd.DataFrame([
    {
        'Reference': 'HGNC complete set',
        'Path': str(hgnc_path),
        'Size_MB': hgnc_path.stat().st_size / 1e6,
    },
    {
        'Reference': 'STRING protein info',
        'Path': str(string_info_path),
        'Size_MB': string_info_path.stat().st_size / 1e6,
    },
    {
        'Reference': 'STRING links',
        'Path': str(string_links_path),
        'Size_MB': string_links_path.stat().st_size / 1e6,
    },
    {
        'Reference': 'GPL570 annotation',
        'Path': str(gpl570_path),
        'Size_MB': gpl570_path.stat().st_size / 1e6,
    },
])
reference_summary.to_csv(
    TABLE_DIR / 'reference_download_summary.csv',
    index=False
)
display(reference_summary)

finish_stage(token)

## 7. Download and parse all 11 GEO Series Matrix files

In [ ]:
token = start_stage('GEO download and parsing')

all_expression = {}
all_metadata = {}
download_rows = []
parse_rows = []

for row in discovery_table.itertuples(index=False):
    gse = row.GEO_ID
    platform = row.Platform

    series_path = download_first_available(
        geo_series_urls(gse, platform),
        d['geo'] / f'{gse}_series_matrix.txt.gz'
    )
    download_rows.append({
        'GEO_ID': gse,
        'Platform': platform,
        'Series_matrix_path': str(series_path),
        'Bytes': series_path.stat().st_size,
    })

    expression, metadata_dict = parse_series_matrix(series_path)
    metadata = create_sample_metadata(metadata_dict)
    metadata['GEO_ID'] = gse
    metadata['Label'] = metadata.apply(
        lambda sample_row: label_discovery_sample(gse, sample_row),
        axis=1
    )
    metadata = metadata[
        metadata['Label'].isin(['Cancer', 'Normal'])
    ].copy()

    sample_ids = [
        sample_id
        for sample_id in metadata['GSM_ID']
        if sample_id in expression.columns
    ]
    expression = expression[['ID_REF'] + sample_ids].copy()
    metadata = (
        metadata
        .set_index('GSM_ID')
        .loc[sample_ids]
        .reset_index()
    )

    all_expression[gse] = expression
    all_metadata[gse] = metadata

    parse_rows.append({
        'GEO_ID': gse,
        'Status': 'loaded',
        'Samples': len(sample_ids),
        'Cancer': int((metadata['Label'] == 'Cancer').sum()),
        'Normal': int((metadata['Label'] == 'Normal').sum()),
        'Raw_probe_sets': len(expression),
    })

download_summary = pd.DataFrame(download_rows)
parse_summary = pd.DataFrame(parse_rows)

download_summary.to_csv(
    TABLE_DIR / 'geo_download_summary.csv',
    index=False
)
parse_summary.to_csv(
    TABLE_DIR / 'geo_parse_summary.csv',
    index=False
)

display(download_summary)
display(parse_summary)
print('Total parsed samples:', parse_summary['Samples'].sum())

finish_stage(token)

## 8. Inspect the expression scale before probe-to-gene mapping

In [ ]:
def expression_scale_row(gse, expression, threshold):
    sample_columns = [
        column
        for column in expression.columns
        if column != 'ID_REF'
    ]
    numeric = (
        expression[sample_columns]
        .apply(pd.to_numeric, errors='coerce')
    )
    values = numeric.to_numpy(dtype=float).ravel()
    values = values[np.isfinite(values)]

    quantiles = np.percentile(
        values,
        [0, 1, 25, 50, 75, 99, 100]
    )
    minimum, q01, q25, median, q75, q99, maximum = quantiles
    apply_log = bool(
        minimum >= 0
        and maximum > float(threshold)
    )

    return {
        'GEO_ID': gse,
        'N_values': len(values),
        'Minimum': minimum,
        'Q01': q01,
        'Q25': q25,
        'Median': median,
        'Q75': q75,
        'Q99': q99,
        'Maximum': maximum,
        'Has_negative_values': bool(minimum < 0),
        'Log2_threshold': float(threshold),
        'Apply_log2_x_plus_1': apply_log,
        'Decision': (
            'Apply log2(x+1)'
            if apply_log
            else 'Already approximately log2 scale'
        ),
    }

scale_check = pd.DataFrame([
    expression_scale_row(
        gse,
        expression,
        cfg['preprocessing']['log2_threshold']
    )
    for gse, expression in all_expression.items()
])

scale_check.to_csv(
    TABLE_DIR / 'expression_scale_check_before_mapping.csv',
    index=False
)
scale_check[
    [
        'GEO_ID', 'Maximum', 'Log2_threshold',
        'Apply_log2_x_plus_1', 'Decision'
    ]
].to_csv(
    TABLE_DIR / 'expression_scale_decision_summary.csv',
    index=False
)

display(scale_check)

## 9. Conditional log transformation, probe-to-gene mapping, and explicit common-gene intersection

This corrects a major ambiguity in the earlier notebook: the integrated matrix is built from the **intersection** of genes available in every discovery cohort, not from an implicit union produced by `pandas.concat`.

In [ ]:
token = start_stage(
    'Probe mapping, common-gene intersection, normalization, and ComBat'
)

mapped_matrices = {}
mapped_metadata = {}
cohort_qc_rows = []

for gse, expression in all_expression.items():
    transformed_expression, log_applied = conditional_log2_probe_table(
        expression,
        float(cfg['preprocessing']['log2_threshold'])
    )
    gene_matrix = highest_variance_probe_per_gene(
        transformed_expression,
        probe_map
    )

    sample_ids = all_metadata[gse]['GSM_ID'].tolist()
    gene_matrix = gene_matrix.loc[sample_ids]

    mapped_matrices[gse] = gene_matrix
    mapped_metadata[gse] = all_metadata[gse].copy()

    cohort_qc_rows.append({
        'GEO_ID': gse,
        'Samples': gene_matrix.shape[0],
        'Genes_after_probe_mapping': gene_matrix.shape[1],
        'Log2_applied': bool(log_applied),
        'Missing_values_after_mapping': int(
            gene_matrix.isna().sum().sum()
        ),
    })

common_genes = sorted(
    set.intersection(
        *[
            set(matrix.columns)
            for matrix in mapped_matrices.values()
        ]
    )
)

X_common = pd.concat([
    mapped_matrices[gse][common_genes]
    for gse in discovery_table['GEO_ID']
])
metadata_all = pd.concat([
    mapped_metadata[gse]
    for gse in discovery_table['GEO_ID']
], ignore_index=True)

metadata_all = (
    metadata_all
    .set_index('GSM_ID')
    .loc[X_common.index]
    .reset_index()
)
y_all = binary_labels(metadata_all['Label'])

print('Common-gene matrix:', X_common.shape)
print('Metadata:', metadata_all.shape)
print('Cancer:', int((y_all == 1).sum()))
print('Normal:', int((y_all == 0).sum()))
print('Missing values:', int(X_common.isna().sum().sum()))

X_qn = quantile_normalize_samples(X_common)
X_harmonized, combat_estimates = combat_harmonize(
    X_qn,
    metadata_all['GEO_ID'],
    y_all,
    preserve_class=bool(
        cfg['preprocessing']['combat_preserve_class']
    )
)
X_final, variance_cutoff = filter_bottom_variance(
    X_harmonized,
    float(cfg['preprocessing']['variance_bottom_fraction'])
)

preprocessing_stages = pd.DataFrame([
    {
        'Stage': 'Common-gene intersection after probe mapping',
        'Samples': X_common.shape[0],
        'Genes': X_common.shape[1],
    },
    {
        'Stage': 'Joint quantile normalization',
        'Samples': X_qn.shape[0],
        'Genes': X_qn.shape[1],
    },
    {
        'Stage': 'ComBat harmonization with class preservation',
        'Samples': X_harmonized.shape[0],
        'Genes': X_harmonized.shape[1],
    },
    {
        'Stage': 'Bottom-10% variance filtering',
        'Samples': X_final.shape[0],
        'Genes': X_final.shape[1],
    },
])

pd.DataFrame(cohort_qc_rows).to_csv(
    TABLE_DIR / 'discovery_cohort_QC.csv',
    index=False
)
preprocessing_stages.to_csv(
    TABLE_DIR / 'preprocessing_stage_dimensions.csv',
    index=False
)
preprocessing_stages.to_csv(
    TABLE_DIR / 'gene_count_summary.csv',
    index=False
)

X_final.to_csv(
    TABLE_DIR / 'harmonized_expression_matrix.csv.gz',
    compression='gzip',
    index=True,
    index_label='GSM_ID'
)
metadata_all.to_csv(
    TABLE_DIR / 'harmonized_metadata.csv',
    index=False
)

display(pd.DataFrame(cohort_qc_rows))
display(preprocessing_stages)

finish_stage(token)

## 10. Preprocessing QC, missing-value audit, PCA, correlation, and outlier review

In [ ]:
token = start_stage('Preprocessing QC')

pca_before, variance_before = pca_table(
    X_common,
    metadata_all,
    'Before harmonization',
    RANDOM_STATE
)
pca_after, variance_after = pca_table(
    X_harmonized,
    metadata_all,
    'After harmonization',
    RANDOM_STATE
)
pca_final, variance_final = pca_table(
    X_final,
    metadata_all,
    'Final variance-filtered matrix',
    RANDOM_STATE
)

distribution_before = expression_distribution_summary(
    X_common,
    metadata_all,
    'Before harmonization'
)
distribution_after = expression_distribution_summary(
    X_harmonized,
    metadata_all,
    'After harmonization'
)
distribution_final = expression_distribution_summary(
    X_final,
    metadata_all,
    'Final variance-filtered matrix'
)
distribution_all = pd.concat(
    [
        distribution_before,
        distribution_after,
        distribution_final,
    ],
    ignore_index=True
)

qc_table, correlation_matrix = sample_correlation_audit(
    X_final,
    metadata_all,
    pca_final,
    float(cfg['quality_control']['robust_z_threshold'])
)

missing_summary = missing_value_summary({
    'Common_gene_matrix': X_common,
    'Quantile_normalized': X_qn,
    'ComBat_harmonized': X_harmonized,
    'Variance_filtered': X_final,
})

qc_decision = qc_table.copy()
qc_decision['Decision'] = np.where(
    qc_decision['Audit_flag'],
    (
        'Flagged for audit; retained unless a technical '
        'inconsistency is confirmed'
    ),
    'Retained'
)
qc_decision['Removed'] = False

distribution_all.to_csv(
    TABLE_DIR / 'sample_distribution_summary_all_stages.csv',
    index=False
)
missing_summary.to_csv(
    TABLE_DIR / 'missing_value_and_integrity_summary.csv',
    index=False
)
qc_table.to_csv(
    TABLE_DIR / 'sample_QC_audit_metrics.csv',
    index=False
)
qc_decision.to_csv(
    TABLE_DIR / 'qc_flagged_samples_decision_table.csv',
    index=False
)
qc_decision.groupby(
    ['GEO_ID', 'Label', 'Decision'],
    as_index=False
).size().rename(
    columns={'size': 'Count'}
).to_csv(
    TABLE_DIR / 'qc_flagged_samples_decision_summary.csv',
    index=False
)

display(missing_summary)
display(qc_decision.head(20))
print('Combined audit flags:', int(qc_table['Audit_flag'].sum()))
print('Samples removed automatically: 0')

finish_stage(token)

## 11. Preprocessing figures

In [ ]:
variance_summary = pd.concat(
    [variance_before, variance_after, variance_final],
    ignore_index=True
)
variance_summary.to_csv(
    TABLE_DIR / 'pca_variance_summary.csv',
    index=False
)

plot_pca_before_after(
    pca_before,
    pca_after,
    pd.concat([variance_before, variance_after], ignore_index=True),
    FIGURE_DIR / 'Figure_2_PCA_before_after_harmonization.png'
)
plot_pca_before_after(
    pca_before,
    pca_after,
    pd.concat([variance_before, variance_after], ignore_index=True),
    FIGURE_DIR / 'Figure_2_PCA_before_after_harmonization.pdf'
)

plot_expression_distributions(
    distribution_all,
    SUPP_FIGURE_DIR / 'Figure_S1_Expression_distributions.png'
)
plot_expression_distributions(
    distribution_all,
    SUPP_FIGURE_DIR / 'Figure_S1_Expression_distributions.pdf'
)

plot_missing_values(
    missing_summary,
    SUPP_FIGURE_DIR / 'Figure_S2_Missing_value_audit.png'
)
plot_correlation_heatmap(
    correlation_matrix,
    metadata_all,
    SUPP_FIGURE_DIR / 'Figure_S3_Sample_correlation_heatmap.png'
)
plot_outlier_audit(
    qc_table,
    SUPP_FIGURE_DIR / 'Figure_S4_Sample_outlier_audit.png'
)
plot_preprocessing_qc_overview(
    distribution_all,
    missing_summary,
    variance_summary,
    qc_table,
    SUPP_FIGURE_DIR / 'Figure_S5_Preprocessing_QC_overview.png'
)

from IPython.display import Image, display as show
show(
    Image(
        filename=str(
            FIGURE_DIR /
            'Figure_2_PCA_before_after_harmonization.png'
        )
    )
)

## 12. Stratified development–held-out split and five-fold assignments

In [ ]:
token = start_stage('Development-heldout split')

train_indices, test_indices = train_test_split(
    np.arange(len(X_final)),
    test_size=float(cfg['project']['test_size']),
    stratify=y_all,
    random_state=RANDOM_STATE
)

X_train = X_final.iloc[train_indices].copy()
X_test = X_final.iloc[test_indices].copy()
y_train = y_all[train_indices]
y_test = y_all[test_indices]

metadata_train = (
    metadata_all
    .iloc[train_indices]
    .reset_index(drop=True)
)
metadata_test = (
    metadata_all
    .iloc[test_indices]
    .reset_index(drop=True)
)

split_assignments = pd.concat([
    metadata_train.assign(
        Set='Model-development',
        Label_binary=y_train
    ),
    metadata_test.assign(
        Set='Post-harmonization held-out',
        Label_binary=y_test
    ),
], ignore_index=True)

split_assignments.to_csv(
    TABLE_DIR / 'train_test_split_assignments.csv',
    index=False
)
split_summary = (
    split_assignments
    .groupby(['Set', 'Label'], as_index=False)
    .size()
    .rename(columns={'size': 'Samples'})
)
split_summary.to_csv(
    TABLE_DIR / 'train_test_class_distribution.csv',
    index=False
)

display(split_summary)
print('Development matrix:', X_train.shape)
print('Held-out matrix:', X_test.shape)

finish_stage(token)

## 13. Build the high-confidence STRING PPI degree table

In [ ]:
token = start_stage('STRING PPI construction')

protein_to_gene = read_string_protein_to_gene(
    string_info_path,
    resolver
)
string_edges, ppi_degree = build_gene_edges_and_degree(
    string_links_path,
    protein_to_gene,
    X_final.columns,
    required_score=int(cfg['ppi']['required_score']),
    chunk_size=int(cfg['ppi']['chunk_size'])
)
ppi_rank_table = prepare_ppi_rank_table(
    ppi_degree,
    X_final.columns
)

string_edges.to_csv(
    TABLE_DIR / 'STRING_gene_edges_eligible_genes.csv',
    index=False
)
ppi_degree.to_csv(
    TABLE_DIR / 'ppi_degree_table.csv',
    index=False
)
ppi_rank_table.to_csv(
    TABLE_DIR / 'ppi_rank_table_training_genes.csv',
    index=False
)

print('Eligible genes:', X_final.shape[1])
print('Retained high-confidence gene edges:', len(string_edges))
display(ppi_rank_table.head(30))

finish_stage(token)

## 14. Five-fold training-restricted feature selection and classification

All supervised operations below are recomputed using only the fold-training samples.  
The held-out fold is used only for prediction.

In [ ]:
# ============================================================
# 14. Five-fold training-restricted feature selection,
#     classification, and panel-size sensitivity
#
# Replacement cell:
# - displays detailed progress;
# - displays temporary metrics after every fold;
# - saves a checkpoint after every completed fold;
# - recreates all variables required by the next stability cell.
# ============================================================

import gc
import time
import numpy as np
import pandas as pd
from IPython.display import display

# ------------------------------------------------------------
# A. Check that all previous cells are still active
# ------------------------------------------------------------
required_objects = [
    "X_train",
    "y_train",
    "metadata_train",
    "ppi_degree",
    "ppi_rank_table",
    "cfg",
    "TABLE_DIR",
    "FINAL_K",
    "K_VALUES",
    "CV_FOLDS",
    "RANDOM_STATE",
    "rankings",
    "panels",
    "_long_panels",
    "select_top_k",
    "create_models",
    "fit_predict_panels",
    "metrics_from_predictions",
    "summarize_cv",
    "start_stage",
    "finish_stage",
]

missing_objects = [
    name for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Runtime kehilangan objek dari cell sebelumnya. "
        "Jalankan ulang notebook sampai selesai bagian "
        "'Build the high-confidence STRING PPI degree table', "
        "kemudian kembali ke cell ini.\n\nObjek yang belum ada: "
        + ", ".join(missing_objects)
    )

# ------------------------------------------------------------
# B. Start a completely fresh CV execution
# ------------------------------------------------------------
token = start_stage(
    "Five-fold feature selection, classification, "
    "and panel-size sensitivity"
)

models = create_models(cfg)
default_threshold = float(
    cfg["models"]["default_decision_threshold"]
)

splitter = StratifiedKFold(
    n_splits=CV_FOLDS,
    shuffle=True,
    random_state=RANDOM_STATE,
)
cv_splits = list(
    splitter.split(X_train, y_train)
)

# Reset all objects, so interrupted results are not duplicated.
fold_rankings = {
    method: {}
    for method in cfg["feature_selection"]["methods"]
}
prediction_frames = []
ppi_prediction_frames = []
fold_panel_rows = []
fold_membership_rows = []
validation_assignment_rows = []

checkpoint_dir = (
    TABLE_DIR /
    "cv_checkpoints"
)
checkpoint_dir.mkdir(
    parents=True,
    exist_ok=True
)

print(
    f"Development samples : {len(X_train)}\n"
    f"Eligible genes      : {X_train.shape[1]}\n"
    f"CV folds            : {CV_FOLDS}\n"
    f"Panel sizes         : {K_VALUES}\n"
    f"Selectors           : "
    f"{cfg['feature_selection']['methods']}\n"
    f"Classifiers         : {list(models.keys())}\n"
    f"Primary panel       : k={FINAL_K}\n",
    flush=True,
)

# ------------------------------------------------------------
# C. Run every fold
# ------------------------------------------------------------
for fold_number, (
    fit_indices,
    validation_indices,
) in enumerate(
    cv_splits,
    start=1,
):
    fold_start = time.perf_counter()

    print("\n" + "=" * 78, flush=True)
    print(f"FOLD {fold_number}/{CV_FOLDS}", flush=True)
    print(f"Training samples   : {len(fit_indices)}", flush=True)
    print(f"Validation samples : {len(validation_indices)}", flush=True)
    print("=" * 78, flush=True)

    print(
        f"[Fold {fold_number}] "
        "Calculating limma, NIBFS, DEG-only, "
        "mRMR, and LASSO rankings...",
        flush=True,
    )

    ranking_tables = rankings(
        X_train.iloc[fit_indices],
        y_train[fit_indices],
        ppi_degree,
        cfg,
    )

    print(
        f"[Fold {fold_number}] "
        "All feature rankings completed.",
        flush=True,
    )

    for method, ranking_table in ranking_tables.items():
        fold_rankings[method][fold_number] = ranking_table.copy()
        ranking_table.to_csv(
            TABLE_DIR /
            f"fold{fold_number}_{method.replace(' ', '_')}_ranking.csv",
            index=False,
        )

    fold_prediction_parts = []
    fold_ppi_prediction_parts = []

    for k in K_VALUES:
        k_start = time.perf_counter()

        print(
            f"\n[Fold {fold_number}] "
            f"Running k={k}: main selectors with LR, RF, and LightGBM...",
            flush=True,
        )

        selected_panels = panels(ranking_tables, k)
        fold_panel_rows.extend(
            _long_panels(fold_number, k, selected_panels)
        )

        fold_predictions, _ = fit_predict_panels(
            X_train.iloc[fit_indices],
            y_train[fit_indices],
            X_train.iloc[validation_indices],
            y_train[validation_indices],
            selected_panels,
            models,
            f"CV fold {fold_number}",
        )
        fold_predictions["Fold"] = fold_number
        prediction_frames.append(fold_predictions)
        fold_prediction_parts.append(fold_predictions)

        print(
            f"[Fold {fold_number}] Main selectors at k={k} completed.",
            flush=True,
        )

        print(
            f"[Fold {fold_number}] Running auxiliary PPI-only at k={k}...",
            flush=True,
        )

        ppi_only_genes = select_top_k(
            ppi_rank_table,
            k,
            "Rank_topo",
        )

        ppi_predictions, _ = fit_predict_panels(
            X_train.iloc[fit_indices],
            y_train[fit_indices],
            X_train.iloc[validation_indices],
            y_train[validation_indices],
            {"PPI-only": ppi_only_genes},
            models,
            f"CV fold {fold_number}",
        )
        ppi_predictions["Fold"] = fold_number
        ppi_prediction_frames.append(ppi_predictions)
        fold_ppi_prediction_parts.append(ppi_predictions)

        k_seconds = time.perf_counter() - k_start
        print(
            f"[Fold {fold_number}] k={k} completed in "
            f"{k_seconds / 60:.2f} minutes.",
            flush=True,
        )

        if int(k) == int(FINAL_K):
            primary_fold_metrics = metrics_from_predictions(
                fold_predictions,
                default_threshold=default_threshold,
            )

            preview_columns = [
                column
                for column in [
                    "Feature_selection_method",
                    "Classifier",
                    "k",
                    "ROC_AUC",
                    "Accuracy",
                    "Balanced_accuracy",
                    "F1",
                    "MCC",
                ]
                if column in primary_fold_metrics.columns
            ]

            print(
                f"\nTemporary metrics — Fold {fold_number}, k={FINAL_K}",
                flush=True,
            )
            display(
                primary_fold_metrics[preview_columns].sort_values(
                    ["Feature_selection_method", "Classifier"]
                )
            )

    for index in fit_indices:
        row = metadata_train.iloc[index]
        fold_membership_rows.append({
            "Fold": fold_number,
            "Subset": "Training",
            "GSM_ID": row.GSM_ID,
            "GEO_ID": row.GEO_ID,
            "Label": row.Label,
        })

    for index in validation_indices:
        row = metadata_train.iloc[index]
        fold_membership_rows.append({
            "Fold": fold_number,
            "Subset": "Validation",
            "GSM_ID": row.GSM_ID,
            "GEO_ID": row.GEO_ID,
            "Label": row.Label,
        })
        validation_assignment_rows.append({
            "GSM_ID": row.GSM_ID,
            "GEO_ID": row.GEO_ID,
            "Label": row.Label,
            "Label_binary": int(y_train[index]),
            "Validation_fold": fold_number,
        })

    fold_prediction_checkpoint = pd.concat(
        fold_prediction_parts,
        ignore_index=True,
    )
    fold_ppi_checkpoint = pd.concat(
        fold_ppi_prediction_parts,
        ignore_index=True,
    )

    fold_prediction_checkpoint.to_csv(
        checkpoint_dir / f"fold{fold_number}_main_predictions.csv",
        index=False,
    )
    fold_ppi_checkpoint.to_csv(
        checkpoint_dir / f"fold{fold_number}_ppi_only_predictions.csv",
        index=False,
    )

    pd.DataFrame(fold_panel_rows).to_csv(
        checkpoint_dir / "selected_panels_completed_folds.csv",
        index=False,
    )
    pd.DataFrame(fold_membership_rows).to_csv(
        checkpoint_dir / "fold_membership_completed_folds.csv",
        index=False,
    )
    pd.DataFrame(validation_assignment_rows).to_csv(
        checkpoint_dir / "validation_assignments_completed_folds.csv",
        index=False,
    )

    completed_main_predictions = pd.concat(
        prediction_frames,
        ignore_index=True,
    )
    completed_main_metrics = metrics_from_predictions(
        completed_main_predictions,
        default_threshold=default_threshold,
    )
    completed_main_metrics["Fold"] = (
        completed_main_metrics["Dataset"]
        .str.extract(r"(\d+)")[0]
        .astype(int)
    )

    completed_main_predictions.to_csv(
        checkpoint_dir / "all_completed_fold_predictions.csv",
        index=False,
    )
    completed_main_metrics.to_csv(
        checkpoint_dir / "all_completed_fold_metrics.csv",
        index=False,
    )

    fold_seconds = time.perf_counter() - fold_start
    print(
        f"\nFOLD {fold_number}/{CV_FOLDS} COMPLETED in "
        f"{fold_seconds / 60:.2f} minutes.",
        flush=True,
    )
    print("Checkpoint saved to:", checkpoint_dir, flush=True)

    gc.collect()

# ------------------------------------------------------------
# D. Combine the five completed folds
# ------------------------------------------------------------
print("\n" + "=" * 78, flush=True)
print(
    "ALL FIVE FOLDS COMPLETED. "
    "Combining predictions and calculating final summaries...",
    flush=True,
)
print("=" * 78, flush=True)

cv_predictions = pd.concat(
    prediction_frames,
    ignore_index=True,
)
cv_metrics = metrics_from_predictions(
    cv_predictions,
    default_threshold=default_threshold,
)
cv_metrics["Fold"] = (
    cv_metrics["Dataset"]
    .str.extract(r"(\d+)")[0]
    .astype(int)
)
cv_summary = summarize_cv(cv_metrics)

ppi_cv_predictions = pd.concat(
    ppi_prediction_frames,
    ignore_index=True,
)
ppi_cv_metrics = metrics_from_predictions(
    ppi_cv_predictions,
    default_threshold=default_threshold,
)
ppi_cv_metrics["Fold"] = (
    ppi_cv_metrics["Dataset"]
    .str.extract(r"(\d+)")[0]
    .astype(int)
)
ppi_cv_summary = summarize_cv(ppi_cv_metrics)

fold_panels = pd.DataFrame(fold_panel_rows)
fold_membership = pd.DataFrame(fold_membership_rows)
fold_assignments = (
    pd.DataFrame(validation_assignment_rows)
    .sort_values("GSM_ID")
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# E. Save the official complete CV outputs
# ------------------------------------------------------------
cv_predictions.to_csv(
    TABLE_DIR / "cross_validated_predictions_all_k.csv",
    index=False,
)
cv_metrics.to_csv(
    TABLE_DIR / "cross_validated_performance_by_fold_all_k.csv",
    index=False,
)
cv_summary.to_csv(
    TABLE_DIR / "cross_validated_performance_summary_all_k.csv",
    index=False,
)

cv_metrics[cv_metrics["k"] == FINAL_K].to_csv(
    TABLE_DIR / f"cross_validated_performance_by_fold_k{FINAL_K}.csv",
    index=False,
)
cv_summary[cv_summary["k"] == FINAL_K].to_csv(
    TABLE_DIR / f"cross_validated_performance_summary_k{FINAL_K}.csv",
    index=False,
)

ppi_cv_predictions.to_csv(
    TABLE_DIR / "PPI_only_cross_validated_predictions_all_k.csv",
    index=False,
)
ppi_cv_metrics.to_csv(
    TABLE_DIR / "PPI_only_cross_validated_performance_by_fold_all_k.csv",
    index=False,
)
ppi_cv_summary.to_csv(
    TABLE_DIR / "PPI_only_cross_validated_performance_summary_all_k.csv",
    index=False,
)

fold_panels.to_csv(
    TABLE_DIR / "foldwise_selected_gene_panels_all_k.csv",
    index=False,
)
fold_panels[fold_panels["k"] == FINAL_K].to_csv(
    TABLE_DIR / f"foldwise_selected_gene_panels_k{FINAL_K}.csv",
    index=False,
)
fold_membership.to_csv(
    TABLE_DIR / "fold_membership_all_folds.csv",
    index=False,
)
fold_assignments.to_csv(
    TABLE_DIR / "fold_assignments.csv",
    index=False,
)

# ------------------------------------------------------------
# F. Display the final primary-k results
# ------------------------------------------------------------
final_primary_summary = (
    cv_summary[cv_summary["k"] == FINAL_K]
    .sort_values(["Feature_selection_method", "Classifier"])
    .reset_index(drop=True)
)

print(f"\nFINAL FIVE-FOLD SUMMARY — k={FINAL_K}", flush=True)
display(final_primary_summary)

print("\nPrimary-k fold-level metrics", flush=True)
display(
    cv_metrics[cv_metrics["k"] == FINAL_K].sort_values(
        ["Fold", "Feature_selection_method", "Classifier"]
    )
)

print("\nCV cell completed successfully.", flush=True)
print("Official outputs saved in:", TABLE_DIR, flush=True)

finish_stage(token)


## 15. Pairwise Jaccard stability, selection frequency, and statistical tests

In [ ]:
token = start_stage('Stability and statistical tests')

rank_columns = {
    'NIBFS': 'Rank_NIBFS',
    'DEG-only': 'Rank_stat',
    'mRMR': 'Selection_Order',
    'LASSO': 'Rank_LASSO',
}

pairwise_frames = []
frequency_frames = []
stability_rows = []

for k in K_VALUES:
    for method, fold_tables in fold_rankings.items():
        fold_gene_sets = {
            fold: select_top_k(
                ranking_table,
                k,
                rank_columns[method]
            )
            for fold, ranking_table in fold_tables.items()
        }
        pairwise_table, frequency_table, stability_summary = (
            pairwise_jaccard_summary(
                fold_gene_sets,
                method,
                k
            )
        )
        frequency_table['Method'] = method
        frequency_table['k'] = k

        pairwise_frames.append(pairwise_table)
        frequency_frames.append(frequency_table)
        stability_rows.append(stability_summary)

pairwise_jaccard = pd.concat(
    pairwise_frames,
    ignore_index=True
)
selection_frequency = pd.concat(
    frequency_frames,
    ignore_index=True
)
stability_summary = (
    pd.DataFrame(stability_rows)
    .sort_values(
        ['k', 'Mean_Jaccard'],
        ascending=[True, False]
    )
)

pairwise_jaccard.to_csv(
    TABLE_DIR / 'stability_pairwise_jaccard.csv',
    index=False
)
selection_frequency.to_csv(
    TABLE_DIR / 'stability_gene_frequency.csv',
    index=False
)
stability_summary.to_csv(
    TABLE_DIR / 'stability_summary.csv',
    index=False
)
stability_summary[
    stability_summary['k'] == FINAL_K
].to_csv(
    TABLE_DIR / f'stability_summary_k{FINAL_K}.csv',
    index=False
)

stability_global_test, stability_posthoc = (
    stability_statistical_tests(
        pairwise_jaccard,
        FINAL_K
    )
)
cv_auc_tests = cv_auc_statistical_tests(
    cv_metrics,
    FINAL_K
)

stability_global_test.to_csv(
    TABLE_DIR / 'stability_Friedman_test.csv',
    index=False
)
stability_posthoc.to_csv(
    TABLE_DIR / 'stability_posthoc_Wilcoxon_BH.csv',
    index=False
)
cv_auc_tests.to_csv(
    TABLE_DIR / 'CV_ROCAUC_Wilcoxon_BH.csv',
    index=False
)

display(
    stability_summary[
        stability_summary['k'] == FINAL_K
    ]
)
display(stability_global_test)
display(stability_posthoc)
display(cv_auc_tests)

finish_stage(token)

## 16. Recompute rankings on the complete development set and freeze the top-k panel

In [ ]:
token = start_stage('Full-development rankings and frozen panel')

final_rankings = rankings(
    X_train,
    y_train,
    ppi_degree,
    cfg
)
final_panels = {
    k: panels(final_rankings, k)
    for k in K_VALUES
}

for method, ranking_table in final_rankings.items():
    ranking_table.to_csv(
        TABLE_DIR /
        f'full_training_{method.replace(" ", "_")}_ranking.csv',
        index=False
    )

final_panel = final_rankings['NIBFS'].head(FINAL_K).copy()
frequency_map = (
    selection_frequency[
        (selection_frequency['Method'] == 'NIBFS')
        & (selection_frequency['k'] == FINAL_K)
    ]
    .set_index('Gene')['Fold_Frequency']
)
final_panel['Fold_frequency'] = (
    final_panel['Gene']
    .map(frequency_map)
    .fillna(0)
    .astype(int)
)
final_panel.to_csv(
    TABLE_DIR / f'final_NIBFS_gene_panel_k{FINAL_K}.csv',
    index=False
)
_full_panels(final_panels).to_csv(
    TABLE_DIR / 'final_feature_panels_full_development_all_k.csv',
    index=False
)

volcano_table = final_rankings['DEG-only'].copy()
volcano_table['Descriptive_DEG'] = (
    (volcano_table['FDR'] <= 0.05)
    & (volcano_table['logFC'].abs() > 1)
)
volcano_table[f'Final_NIBFS_k{FINAL_K}'] = (
    volcano_table['Gene'].isin(
        final_panels[FINAL_K]['NIBFS']
    )
)
volcano_table.to_csv(
    TABLE_DIR / 'volcano_limma_full_training_table.csv',
    index=False
)

display(final_panel)

finish_stage(token)

## 17. Internal held-out evaluation, OOF-derived thresholds, bootstrap confidence intervals, and calibration

In [ ]:
token = start_stage('Internal held-out evaluation')

oof_thresholds = derive_youden_thresholds(
    cv_predictions,
    method='NIBFS',
    k=FINAL_K
)
oof_thresholds.to_csv(
    TABLE_DIR / 'discovery_OOF_Youden_thresholds.csv',
    index=False
)

heldout_predictions, heldout_models = fit_predict_panels(
    X_train,
    y_train,
    X_test,
    y_test,
    final_panels[FINAL_K],
    models,
    'Post-harmonization held-out',
)

heldout_default = metrics_from_predictions(
    heldout_predictions,
    default_threshold=default_threshold,
    threshold_source='Default 0.5'
).assign(Evaluation_rule='Default')

heldout_transferred = metrics_from_predictions(
    heldout_predictions,
    thresholds=oof_thresholds,
    default_threshold=default_threshold
).assign(Evaluation_rule='OOF-transferred')

heldout_metrics = pd.concat(
    [heldout_default, heldout_transferred],
    ignore_index=True
)
heldout_metrics = add_auc_confidence_intervals(
    heldout_metrics,
    heldout_predictions,
    int(cfg['external_validation']['bootstrap_iterations']),
    RANDOM_STATE,
)

heldout_calibration = calibration_table(
    heldout_predictions
)

heldout_predictions.to_csv(
    TABLE_DIR / 'heldout_predictions.csv',
    index=False
)
heldout_metrics.to_csv(
    TABLE_DIR / 'heldout_metrics_default_and_transferred.csv',
    index=False
)
heldout_calibration.to_csv(
    TABLE_DIR / 'heldout_calibration_curve.csv',
    index=False
)

for (method, classifier), model in heldout_models.items():
    if method == 'NIBFS':
        joblib.dump(
            model,
            MODEL_DIR /
            f'{classifier}_full_development_k{FINAL_K}.joblib'
        )

display(oof_thresholds)
display(heldout_metrics)

finish_stage(token)

## 18. Rank-weight sensitivity at α = 0.25, 0.50, and 0.75

In [ ]:
token = start_stage('Rank-weight sensitivity')

run_weight_sensitivity(
    RUN_DIR,
    primary_k=FINAL_K,
    alphas=tuple(
        map(
            float,
            cfg['rank_weight_sensitivity']['statistical_weights']
        )
    ),
    random_state=RANDOM_STATE
)

weight_files = sorted(
    TABLE_DIR.glob('*weight*sensitiv*.csv')
)
for weight_file in weight_files:
    print('\n', weight_file.name)
    display(pd.read_csv(weight_file).head(50))

finish_stage(token)

## 19. Functional enrichment and selected-panel STRING network

In [ ]:
token = start_stage(
    'Functional enrichment and selected-panel PPI'
)

frozen_genes = final_panels[FINAL_K]['NIBFS']
panel_edges = final_panel_subnetwork(
    string_edges,
    frozen_genes
)
panel_edges.to_csv(
    TABLE_DIR /
    f'biological_validation_STRING_edges_k{FINAL_K}.csv',
    index=False
)

network_summary, centrality_table = plot_ppi_network(
    frozen_genes,
    panel_edges,
    final_rankings['NIBFS'],
    FIGURE_DIR /
    f'Figure_9_PPI_network_k{FINAL_K}.png',
    RANDOM_STATE
)
plot_ppi_network(
    frozen_genes,
    panel_edges,
    final_rankings['NIBFS'],
    FIGURE_DIR /
    f'Figure_9_PPI_network_k{FINAL_K}.pdf',
    RANDOM_STATE
)

network_summary.to_csv(
    TABLE_DIR /
    f'biological_validation_STRING_network_summary_k{FINAL_K}.csv',
    index=False
)
centrality_table.to_csv(
    TABLE_DIR /
    f'biological_validation_STRING_centrality_k{FINAL_K}.csv',
    index=False
)

raw_enrichment = run_enrichment(
    frozen_genes,
    X_train.columns.tolist(),
    cfg['biological_interpretation']['enrichment_sources']
)
raw_enrichment.to_csv(
    TABLE_DIR / 'enrichment_gprofiler_raw.csv',
    index=False
)

enrichment_table = standardize_enrichment(
    raw_enrichment,
    FINAL_K
)
enrichment_table.to_csv(
    TABLE_DIR / 'biological_validation_enrichment_all_terms.csv',
    index=False
)

_source_subset(
    enrichment_table,
    'GO:BP'
).to_csv(
    TABLE_DIR / 'biological_validation_GO_BP_enrichment.csv',
    index=False
)
_source_subset(
    enrichment_table,
    'KEGG'
).to_csv(
    TABLE_DIR / 'biological_validation_KEGG_enrichment.csv',
    index=False
)
_source_subset(
    enrichment_table,
    'REAC'
).to_csv(
    TABLE_DIR / 'biological_validation_Reactome_enrichment.csv',
    index=False
)

main_enrichment = (
    enrichment_table
    .sort_values('Adjusted_p_value')
    .groupby('Database', as_index=False)
    .head(10)
)
main_enrichment.to_csv(
    TABLE_DIR / 'biological_validation_main_enrichment_table.csv',
    index=False
)

display(network_summary)
display(centrality_table.head(30))
display(main_enrichment)

finish_stage(token)

## 20. Independent GSE15852 external validation

In [ ]:
token = start_stage('Independent GSE15852 validation')

X_external, y_external, metadata_external = load_external(
    cfg,
    d,
    resolver
)

external_availability = pd.DataFrame({
    'Gene': frozen_genes,
    'Available_in_GSE15852': [
        gene in X_external.columns
        for gene in frozen_genes
    ],
})
external_availability.to_csv(
    TABLE_DIR /
    f'external_GSE15852_gene_availability_k{FINAL_K}.csv',
    index=False
)
missing_external_genes = external_availability.loc[
    ~external_availability['Available_in_GSE15852'],
    'Gene'
].tolist()

if (
    missing_external_genes
    and cfg['external_validation']['require_complete_frozen_panel']
):
    raise ValueError(
        'Frozen panel genes missing from GSE15852: '
        + ', '.join(missing_external_genes)
    )

available_external_genes = external_availability.loc[
    external_availability['Available_in_GSE15852'],
    'Gene'
].tolist()

external_predictions, _ = fit_predict_panels(
    X_train,
    y_train,
    X_external,
    y_external,
    {'NIBFS': available_external_genes},
    models,
    'Independent external GSE15852',
)

external_default = metrics_from_predictions(
    external_predictions,
    default_threshold=default_threshold,
    threshold_source='Default 0.5'
).assign(Evaluation_rule='Default')

external_transferred = metrics_from_predictions(
    external_predictions,
    thresholds=oof_thresholds,
    default_threshold=default_threshold
).assign(Evaluation_rule='OOF-transferred')

external_metrics = pd.concat(
    [external_default, external_transferred],
    ignore_index=True
)
external_metrics = add_auc_confidence_intervals(
    external_metrics,
    external_predictions,
    int(cfg['external_validation']['bootstrap_iterations']),
    RANDOM_STATE,
)

training_logfc = (
    final_rankings['NIBFS']
    .set_index('Gene')['logFC']
)
direction_rows = []

for gene in available_external_genes:
    external_logfc = float(
        X_external.loc[y_external == 1, gene].mean()
        - X_external.loc[y_external == 0, gene].mean()
    )
    discovery_logfc = float(training_logfc.loc[gene])

    direction_rows.append({
        'Gene': gene,
        'Training_logFC': discovery_logfc,
        'External_logFC': external_logfc,
        'Training_direction': (
            'Up' if discovery_logfc > 0 else 'Down'
        ),
        'External_direction': (
            'Up' if external_logfc > 0 else 'Down'
        ),
        'Direction_consistent': bool(
            np.sign(discovery_logfc)
            == np.sign(external_logfc)
        ),
    })

external_direction = pd.DataFrame(direction_rows)
external_direction_summary = pd.DataFrame([{
    'Available_genes': len(external_direction),
    'Direction_consistent': int(
        external_direction['Direction_consistent'].sum()
    ),
    'Direction_discordant': int(
        (~external_direction['Direction_consistent']).sum()
    ),
    'Consistency_fraction': float(
        external_direction['Direction_consistent'].mean()
    ),
}])

external_predictions.to_csv(
    TABLE_DIR / 'external_GSE15852_predictions.csv',
    index=False
)
external_metrics.to_csv(
    TABLE_DIR /
    'external_GSE15852_metrics_default_and_transferred.csv',
    index=False
)
external_direction.to_csv(
    TABLE_DIR /
    'external_GSE15852_direction_consistency.csv',
    index=False
)
external_direction_summary.to_csv(
    TABLE_DIR /
    'external_GSE15852_direction_consistency_summary.csv',
    index=False
)
calibration_table(
    external_predictions
).to_csv(
    TABLE_DIR / 'external_GSE15852_calibration_curve.csv',
    index=False
)

display(external_availability)
display(external_metrics)
display(external_direction_summary)

finish_stage(token)

## 21. Compact and full hierarchical heatmaps

In [ ]:
selected_samples, compact_counts = choose_compact_heatmap_samples(
    metadata_train,
    max_per_cohort_class=int(
        cfg['outputs']['compact_heatmap_max_per_cohort_class']
    ),
    random_state=RANDOM_STATE
)
compact_counts.to_csv(
    TABLE_DIR /
    f'compact_heatmap_selected_sample_counts_k{FINAL_K}.csv',
    index=False
)

compact_X = X_train.loc[
    X_train.index.intersection(selected_samples)
]
compact_metadata = (
    metadata_train
    .set_index('GSM_ID')
    .loc[compact_X.index]
    .reset_index()
)

plot_clustered_heatmap(
    compact_X,
    compact_metadata,
    frozen_genes,
    FIGURE_DIR /
    f'Figure_4_Compact_heatmap_k{FINAL_K}.png',
    title=f'Frozen NIBFS top-{FINAL_K} expression heatmap'
)
plot_clustered_heatmap(
    compact_X,
    compact_metadata,
    frozen_genes,
    FIGURE_DIR /
    f'Figure_4_Compact_heatmap_k{FINAL_K}.pdf',
    title=f'Frozen NIBFS top-{FINAL_K} expression heatmap'
)

sample_order, gene_order, _ = clustered_heatmap_orders(
    compact_X,
    frozen_genes
)
pd.DataFrame({
    'GSM_ID': sample_order
}).to_csv(
    TABLE_DIR /
    f'heatmap_sample_order_k{FINAL_K}.csv',
    index=False
)
pd.DataFrame({
    'Gene': gene_order
}).to_csv(
    TABLE_DIR /
    f'heatmap_gene_order_k{FINAL_K}.csv',
    index=False
)

plot_clustered_heatmap(
    X_train,
    metadata_train,
    frozen_genes,
    SUPP_FIGURE_DIR /
    f'Figure_S_Full_heatmap_k{FINAL_K}.png',
    title=(
        f'Full model-development heatmap, '
        f'top-{FINAL_K}'
    )
)

from IPython.display import Image, display as show
show(
    Image(
        filename=str(
            FIGURE_DIR /
            f'Figure_4_Compact_heatmap_k{FINAL_K}.png'
        )
    )
)

In [ ]:
# ============================================================
# Load biological_figures.py explicitly from the active PACKAGE_DIR
# ============================================================

from pathlib import Path
import sys
import importlib
import importlib.util

BIOLOGICAL_FILE = (
    Path(PACKAGE_DIR)
    / "src"
    / "biological_figures.py"
)

print("PACKAGE_DIR used by the notebook:")
print(PACKAGE_DIR)

print("\nExpected file:")
print(BIOLOGICAL_FILE)

print("\nFile exists?")
print(BIOLOGICAL_FILE.exists())

if not BIOLOGICAL_FILE.exists():
    print("\nContents of the active src directory:")

    src_directory = (
        Path(PACKAGE_DIR)
        / "src"
    )

    for file_path in sorted(
        src_directory.glob("*")
    ):
        print(file_path.name)

    raise FileNotFoundError(
        "biological_figures.py tidak ditemukan pada "
        "PACKAGE_DIR/src yang digunakan notebook. "
        "Kemungkinan file di-upload ke salinan folder yang lain."
    )

# Pastikan package final berada paling depan di Python path
package_path = str(
    Path(PACKAGE_DIR)
)

if package_path in sys.path:
    sys.path.remove(
        package_path
    )

sys.path.insert(
    0,
    package_path
)

# Hapus cache package src lama dari RAM
for module_name in list(
    sys.modules.keys()
):
    if (
        module_name == "src"
        or module_name.startswith("src.")
    ):
        del sys.modules[module_name]

importlib.invalidate_caches()

# Muat biological_figures.py langsung dari lokasi yang pasti
module_spec = (
    importlib.util.spec_from_file_location(
        "src.biological_figures",
        BIOLOGICAL_FILE,
    )
)

if (
    module_spec is None
    or module_spec.loader is None
):
    raise ImportError(
        "Tidak dapat membuat module specification "
        "untuk biological_figures.py."
    )

biological_figures = (
    importlib.util.module_from_spec(
        module_spec
    )
)

sys.modules[
    "src.biological_figures"
] = biological_figures

module_spec.loader.exec_module(
    biological_figures
)

plot_enrichment_dotstyle = (
    biological_figures
    .plot_enrichment_dotstyle
)

plot_biological_interpretation_dotstyle = (
    biological_figures
    .plot_biological_interpretation_dotstyle
)

print(
    "\nSUCCESS: biological_figures.py "
    "sudah berhasil dimuat."
)
print(
    "Module loaded from:",
    biological_figures.__file__,
)

In [ ]:
plot_enrichment_dotstyle(
    enrichment_table,
    FIGURE_DIR
    / f"Figure_8_Functional_enrichment_k{FINAL_K}.png",
    top_n=8,
)

plot_biological_interpretation_dotstyle(
    enrichment_table,
    frozen_genes,
    panel_edges,
    final_rankings["NIBFS"],
    FIGURE_DIR
    / f"Figure_10_Biological_interpretation_k{FINAL_K}.png",
    seed=RANDOM_STATE,
)

from IPython.display import Image, display

display(
    Image(
        filename=str(
            FIGURE_DIR
            / f"Figure_10_Biological_interpretation_k{FINAL_K}.png"
        )
    )
)

print(
    "Improved biological figures regenerated."
)

## 22. Generate the complete main and supplementary figure set

In [ ]:
plot_volcano(
    volcano_table,
    frozen_genes,
    FIGURE_DIR / f'Figure_3_Volcano_k{FINAL_K}.png'
)
plot_volcano(
    volcano_table,
    frozen_genes,
    FIGURE_DIR / f'Figure_3_Volcano_k{FINAL_K}.pdf'
)

plot_stability_composite(
    pairwise_jaccard,
    selection_frequency,
    FINAL_K,
    FIGURE_DIR /
    f'Figure_5_Stability_and_recurrence_k{FINAL_K}.png'
)
plot_gene_occurrence_heatmap(
    fold_panels,
    SUPP_FIGURE_DIR /
    f'Figure_S_Gene_occurrence_k{FINAL_K}.png',
    method='NIBFS',
    k=FINAL_K
)
plot_cv_performance(
    cv_metrics,
    FIGURE_DIR /
    f'Figure_6_CV_performance_k{FINAL_K}.png',
    k=FINAL_K
)
plot_sensitivity(
    stability_summary,
    cv_summary,
    SUPP_FIGURE_DIR /
    'Figure_S_Panel_size_sensitivity.png',
    method='NIBFS'
)
plot_rank_landscape(
    final_rankings['NIBFS'],
    frozen_genes,
    SUPP_FIGURE_DIR /
    f'Figure_S_Rank_landscape_k{FINAL_K}.png'
)
plot_top_panel_barplot(
    final_panel,
    SUPP_FIGURE_DIR /
    f'Figure_S_Final_panel_barplot_k{FINAL_K}.png'
)
plot_roc(
    heldout_predictions,
    SUPP_FIGURE_DIR /
    f'Figure_S_Heldout_ROC_k{FINAL_K}.png',
    title='Post-harmonization held-out ROC'
)
plot_heldout_composite(
    heldout_predictions,
    heldout_calibration,
    SUPP_FIGURE_DIR /
    f'Figure_S_Heldout_composite_k{FINAL_K}.png'
)
plot_external_validation(
    external_predictions,
    external_direction,
    FIGURE_DIR /
    f'Figure_7_External_validation_k{FINAL_K}.png'
)
plot_enrichment(
    enrichment_table,
    FIGURE_DIR /
    f'Figure_8_Functional_enrichment_k{FINAL_K}.png'
)
plot_biological_interpretation(
    enrichment_table,
    frozen_genes,
    panel_edges,
    final_rankings['NIBFS'],
    FIGURE_DIR /
    f'Figure_10_Biological_interpretation_k{FINAL_K}.png',
    seed=RANDOM_STATE
)

figure_inventory = pd.DataFrame({
    'Figure_file': [
        str(path.relative_to(RUN_DIR))
        for path in sorted(
            list((RUN_DIR / 'results').rglob('*.png'))
            + list((RUN_DIR / 'results').rglob('*.pdf'))
        )
    ]
})
figure_inventory.to_csv(
    TABLE_DIR / 'figure_output_inventory.csv',
    index=False
)
display(figure_inventory)

## 23. KM Plotter top-k template and post hoc integration

In [ ]:
# ============================================================
# KM Plotter RFS integration — final frozen top-20
# ============================================================

from pathlib import Path
import shutil
import numpy as np
import pandas as pd

expected_filename = f"KMPlotter_RFS_final_k{FINAL_K}.csv"

# Search the repository manual-input folder first, then the timestamped run folder.
candidate_paths = [
    PACKAGE_DIR / "manual_inputs" / expected_filename,
    RUN_DIR / "manual_inputs" / expected_filename,
]

existing_paths = [
    path
    for path in candidate_paths
    if path.exists()
]

print("KM Plotter file locations:")
for path in candidate_paths:
    print(
        "FOUND :" if path.exists() else "MISSING:",
        path
    )

if not existing_paths:
    raise FileNotFoundError(
        "KM Plotter file not found. "
        f"Pastikan file bernama tepat {expected_filename} "
        "under PACKAGE_DIR/manual_inputs/."
    )

# Gunakan file pertama yang ditemukan
km_source_path = existing_paths[0]

print("\nKM Plotter file used:")
print(km_source_path)

# Tampilkan file mentah agar jelas bahwa nilainya tidak kosong
km_raw = pd.read_csv(km_source_path)

print("\nRaw KM Plotter data:")
display(km_raw)

# Pemeriksaan kolom
required_raw_columns = {
    "Gene",
    "HR",
    "CI_low",
    "CI_high",
    "p",
}

missing_columns = (
    required_raw_columns
    - set(km_raw.columns)
)

if missing_columns:
    raise ValueError(
        "Required KM Plotter columns are missing: "
        + ", ".join(sorted(missing_columns))
    )

# Salin file manual ke folder official run
official_km_path = (
    RUN_DIR
    / "manual_inputs"
    / expected_filename
)
official_km_path.parent.mkdir(
    parents=True,
    exist_ok=True
)

if km_source_path.resolve() != official_km_path.resolve():
    shutil.copy2(
        km_source_path,
        official_km_path
    )

# Baca dan standardisasi menggunakan fungsi paket
km_results = load_kmplotter_csv(
    official_km_path
)

if km_results.empty:
    raise RuntimeError(
        "File KM Plotter ditemukan tetapi tidak menghasilkan data."
    )

# Bersihkan nama gen
km_results["Gene"] = (
    km_results["Gene"]
    .astype(str)
    .str.strip()
)

# Pastikan kolom numerik benar
for column in [
    "Hazard_ratio",
    "CI_low",
    "CI_high",
    "P_value",
]:
    km_results[column] = pd.to_numeric(
        km_results[column],
        errors="coerce"
    )

print("\nMissing values after standardization:")
display(
    km_results[
        [
            "Hazard_ratio",
            "CI_low",
            "CI_high",
            "P_value",
        ]
    ].isna().sum().to_frame(
        "Jumlah_NaN"
    )
)

# Hentikan jika nilai utama masih NaN
numeric_columns = [
    "Hazard_ratio",
    "CI_low",
    "CI_high",
    "P_value",
]

if km_results[numeric_columns].isna().any().any():
    problematic_rows = km_results[
        km_results[numeric_columns]
        .isna()
        .any(axis=1)
    ]

    print("Rows with invalid values:")
    display(problematic_rows)

    raise RuntimeError(
        "KM Plotter contains values that could not be parsed."
    )

# Cocokkan dengan frozen panel
frozen_gene_list = list(
    map(str, frozen_genes)
)
frozen_gene_set = set(
    frozen_gene_list
)
km_gene_set = set(
    km_results["Gene"]
)

missing_km_genes = sorted(
    frozen_gene_set - km_gene_set
)
extra_km_genes = sorted(
    km_gene_set - frozen_gene_set
)

coverage_summary = pd.DataFrame([{
    "Frozen_panel_size": len(frozen_gene_list),
    "KM_rows": len(km_results),
    "Matched_genes": len(
        frozen_gene_set & km_gene_set
    ),
    "Missing_genes": "; ".join(
        missing_km_genes
    ),
    "Extra_genes": "; ".join(
        extra_km_genes
    ),
    "Complete_top20": (
        len(missing_km_genes) == 0
    ),
}])

print("\nKM Plotter coverage:")
display(coverage_summary)

if missing_km_genes:
    raise RuntimeError(
        "KM Plotter input is incomplete. Missing genes: "
        + ", ".join(missing_km_genes)
    )

# Ambil tepat frozen top-20
km_results = km_results[
    km_results["Gene"].isin(
        frozen_gene_set
    )
].copy()

# Urutkan sesuai frozen panel
gene_order = {
    gene: position
    for position, gene in enumerate(
        frozen_gene_list,
        start=1
    )
}

km_results["Frozen_order"] = (
    km_results["Gene"]
    .map(gene_order)
)

km_results = (
    km_results
    .sort_values("Frozen_order")
    .reset_index(drop=True)
)

# Tambahkan hasil signifikansi
km_results["Nominal_significant"] = (
    km_results["P_value"] < 0.05
)

# Simpan tabel standar
standardized_path = (
    TABLE_DIR
    / f"KMPlotter_RFS_standardized_k{FINAL_K}.csv"
)

km_results.to_csv(
    standardized_path,
    index=False
)

# Buat forest plot
forest_png = (
    SUPP_FIGURE_DIR
    / f"Figure_KMPlotter_RFS_forest_k{FINAL_K}.png"
)
forest_pdf = (
    SUPP_FIGURE_DIR
    / f"Figure_KMPlotter_RFS_forest_k{FINAL_K}.pdf"
)

plot_km_forest(
    km_results,
    forest_png
)

plot_km_forest(
    km_results,
    forest_pdf
)

print("\nKM Plotter integration: COMPLETE")
print("Standardized table:", standardized_path)
print("Forest plot PNG:", forest_png)
print("Forest plot PDF:", forest_pdf)

# Display the standardized results.
display(
    km_results[
        [
            "Frozen_order",
            "Gene",
            "Hazard_ratio",
            "CI_low",
            "CI_high",
            "P_value",
            "Nominal_significant",
        ]
    ]
)

In [ ]:
km_manual_path = (
    RUN_DIR /
    cfg['biological_interpretation']['kmplotter_csv']
)
km_template_path = (
    RUN_DIR /
    'manual_inputs' /
    f'KMPlotter_RFS_final_k{FINAL_K}_TEMPLATE.csv'
)
km_template_path.parent.mkdir(
    parents=True,
    exist_ok=True
)

if not km_template_path.exists():
    pd.DataFrame({
        'Gene': frozen_genes,
        'KM_gene': '',
        'Probe': '',
        'HR': np.nan,
        'CI_low': np.nan,
        'CI_high': np.nan,
        'p': np.nan,
        'direction': '',
    }).to_csv(
        km_template_path,
        index=False
    )

km_results = pd.DataFrame()

if km_manual_path.exists():
    km_results = load_kmplotter_csv(km_manual_path)
    missing_km_genes = sorted(
        set(frozen_genes)
        - set(km_results['Gene'].astype(str))
    )
    if missing_km_genes:
        print(
            'KM Plotter input is incomplete. Missing:',
            missing_km_genes
        )
    else:
        km_results = km_results[
            km_results['Gene'].isin(frozen_genes)
        ].copy()
        km_results.to_csv(
            TABLE_DIR /
            f'KMPlotter_RFS_standardized_k{FINAL_K}.csv',
            index=False
        )
        plot_km_forest(
            km_results,
            SUPP_FIGURE_DIR /
            f'Figure_KMPlotter_RFS_forest_k{FINAL_K}.png'
        )
else:
    print(
        'KM Plotter manual input is not complete yet. '
        'Fill this template after the fresh panel is frozen:'
    )
    print(km_template_path)

display(pd.read_csv(km_template_path))

## 24. Freeze RF–LightGBM OOF, held-out, and external inputs for KAN

In [ ]:
kan_bridge = build_kan_bridge(
    cv_predictions,
    heldout_predictions,
    metadata_train,
    metadata_test,
    k=FINAL_K,
    external_predictions=external_predictions,
    external_metadata=metadata_external,
)

kan_bridge['train_oof'].to_csv(
    KAN_DIR / f'KAN_meta_train_OOF_k{FINAL_K}.csv',
    index=False
)
kan_bridge['heldout'].to_csv(
    KAN_DIR / f'KAN_meta_heldout_k{FINAL_K}.csv',
    index=False
)
kan_bridge['external'].to_csv(
    KAN_DIR /
    f'KAN_meta_external_GSE15852_k{FINAL_K}.csv',
    index=False
)
final_panel.to_csv(
    KAN_DIR / f'KAN_frozen_gene_panel_k{FINAL_K}.csv',
    index=False
)
fold_assignments.to_csv(
    KAN_DIR / 'KAN_fold_assignments.csv',
    index=False
)

kan_train = kan_bridge['train_oof']
kan_audit = pd.DataFrame([
    {
        'Check': 'OOF rows',
        'Expected': len(X_train),
        'Observed': len(kan_train),
    },
    {
        'Check': 'Unique OOF samples',
        'Expected': len(X_train),
        'Observed': kan_train['Sample_ID'].nunique(),
    },
    {
        'Check': 'Missing RF probabilities',
        'Expected': 0,
        'Observed': int(kan_train['p_RF'].isna().sum()),
    },
    {
        'Check': 'Missing LightGBM probabilities',
        'Expected': 0,
        'Observed': int(
            kan_train['p_LightGBM'].isna().sum()
        ),
    },
])
kan_audit['Status'] = np.where(
    kan_audit['Expected'] == kan_audit['Observed'],
    'PASS',
    'FAIL'
)
kan_audit.to_csv(
    KAN_DIR / 'KAN_bridge_audit.csv',
    index=False
)

display(kan_audit)
display(kan_train.head())

## Additional analyses

The core notebook ends here. Repeated 10×5 CV, transfer-safe LOCO, RWR-DEG, topology-permutation controls, GSE70947, fold-fitted preprocessing, degree-preserving rewiring, and TCGA-BRCA RNA-seq validation are provided as dedicated source files/notebooks in this repository. See the root `README.md` for the execution map.
